# Exercise 5: Noise in search results

What if your topic model defines a cluster, but semantic search
returns "noisy" results that seem "off-topic". How to handle noisy datasets and
refine the embedding you use to search for topics. 

Goal: Understanding how to detect and address noise in real world datasets

In [1]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO



Even when BERTopic finds a coherent cluster (high c-TF-IDF, tight cosine distances
in the heatmap), running a *search* with the topic embedding can return obviously
off-topic posts.  

What's happening: the topic embedding is the **centroid** (mean) of every document
embedding in the cluster. When the cluster contains a diffuse, generalist
documents, that centroid expands — everything *kind of* matches everything.

## Outline
1. Diagnosing Noise
- Add posts to the dataset
- Before we begin, we are going to add posts to our dataset so that it more
  closely mirrors a real world dataset. First run generate_more_posts.py Then,
  rerun the preprocessing.py and topic_model.py (you can use src/run_pipeline.py)
- View the demo app and review the evaluation results for the topics. Look at
  the topic "space exploration" what do you see? How are the results different? 


Re-run the pre-processing and topic model pipeline. However, change the
INPUT_FILEPATH to NOISE_FILEPATH. 

```
uv run src/run_pipeline.py 
```

2. Localized vs Topic Embeddings
- Diagnosing the problem 
- Implementing "localized" embeddings

# Diagnosing Noise

### Step 1 — Add noise to the dataset and retrain

To reproduce the failure we'll mix the curated `sample_posts.json` with a noise
file (`sample_noise.json`) of off-topic random posts, then retrain.

(Let's run the solutions file, in
case you made changes to processing or topic modeling that might affect the output.)

In `solutions/run_pipeline.py` you'll see: 
```python
NOISE_FILEPATH = REPO / "sample_noise.json"

PreprocessingPipeline(input_filepath=INPUT_FILEPATH, output_filepath=OUTPUT_FILEPATH).run()
```

Swap `INPUT_FILEPATH` for `NOISE_FILEPATH` (or run preprocessing twice — once on each
file) so the noise posts get embedded and stored alongside the curated set. Then:

```bash
uv run python -m solutions.run_pipeline
```

Once it finishes, reload the Demo App, open **Space Exploration**, and notice the
off-topic results that have crept in. We're going to diagnose them below.

### Explore the model output

After retraining, `output/` holds everything we need:

- `topic_information.csv` — size + label of every topic.
- `topic_assignments.csv` — which post is in which topic (and the dreaded `-1`).
- `topic_embeddings.json` — the **centroid** embedding per topic (mean of doc embeddings).
- `topic_keyword_embeddings.json` — the **localized** keyword embedding per topic (the
  embedding of just the top KeyBERT terms; computed in
  `src/topic_model.py:_save_keyword_embeddings`).

In [2]:
import json

import numpy as np
import pandas as pd

from src.config import OUTPUT

topic_info = pd.read_csv(OUTPUT / "topic_information.csv")
assignments = pd.read_csv(OUTPUT / "topic_assignments.csv")
labels = json.loads((OUTPUT / "topic_labels.json").read_text())

print("Topic sizes (note size of -1 outlier bucket):")
print(topic_info[["Topic", "Count", "Name"]].head(15))

outliers = assignments[assignments["topic_id"] == -1]
pct = 100 * len(outliers) / len(assignments) if len(assignments) else 0
print(f"\nOutliers (topic_id == -1): {len(outliers)} of {len(assignments)} ({pct:.1f}%)")

print("\nFirst 5 outlier posts:")
for text in outliers["text"].head(5):
    print(f"  • {text[:120]}")

Topic sizes (note size of -1 outlier bucket):
   Topic  Count                          Name
0      0    101       0_space_life_just_today
1      1     24       1_music_like_listen_let
2      2     18     2_cats_kindness_cat_kitty
3      3     18  3_water_open_cold_open water

Outliers (topic_id == -1): 0 of 161 (0.0%)

First 5 outlier posts:


### Visual: How are the topics clustered? 

Open `output/topic_visualization.html` (the intertopic distance map). Each topic is
a circle in 2-D UMAP space; size is the number of docs assigned.  

- Are topics isolated or overlapping? 

## 2. Topic embedding vs localized embedding

- **Topic Embedding** (mean of doc embeddings) 
    - The geometric centre ("centroid") of the cluster in 384-d space
    - Broadly represents the topic as a whole 
    - If the topic is broad, diffuse, the topic embedding representing it
      encompasses more area. 
- **Localized** (embedding of keywords or distinguishign features) 
    - An embedding generated from the most representative documents or keywords 
    - A more "localized" embedding that represents what makes the topic
      distinctive 
    - If the topic is broad, diffuse, a "localized" embedding can help separate
      posts that are "on topic" from more general posts. 

When the cluster contains posts with general, diffuse content, the topic
embeddings encompasses a larger "area". It matches more documents. 



### Generating a localized embedding

There are a few ways you can create an embedding that is tighter. 

- Use representative documents (ask an LLM to return a summary or keywords)
- Use top-n keywords from c-TF-IDF 
- Identify keywords in the topic that are the most representative of the topic
  because they are most similar to the topic embedding (KeyBERT)

  Here we will use KeyBERT to create topic embeddings that we can use for search.

### KeyBERT representations: the localized embedding source

BERTopic's `KeyBERTInspired` representation 
ranks candidate n-grams from a topic's documents by **cosine similarity to the topic
embedding**. Top-N of those are stored on
`topic_model.topic_aspects_["KeyBERT"]`.

TFI-DF captures the words that distinguish the topic from other topics. 

KeyBERT takes a sample of documents and candidate keywords and measures which is
semantically most meaningful to the topic. 

It does this by calculating the
similarity between candidate keywords and the topic using cosine similarity.
Then it returns the top-n most similar keywords. 


Reference: https://maartengr.github.io/BERTopic/getting_started/representation/representation.html

In [8]:
from bertopic import BERTopic

topic_model = BERTopic.load(str(OUTPUT / "bertopic_model"))

topic_info.head()

# Notice how the Representation kewords generated from TF-IDF are slightly
# different from the ones generated by KeyBERT.

2026-05-05 13:36:32,530 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


,Topic,Count,Name,Representation,KeyBERT,LLM,Representative_Docs
0,0,101,0_space_life_just_today,"['space', 'life', 'just', 'today', 'like', 'mo...","['nasa', 'space', 'astronauts', 'explore', 'ho...",['Space & Life Journeys'],['moon landings of yore seemed distant then to...
1,1,24,1_music_like_listen_let,"['music', 'like', 'listen', 'let', 'new', 'tra...","['melody', 'song', 'songs', 'musics equivalent...",['Music and Feelings'],['ok_hand music makes the journey to a goal wi...
2,2,18,2_cats_kindness_cat_kitty,"['cats', 'kindness', 'cat', 'kitty', 'purr', '...","['kindness', 'purpose cats', 'kitty', 'pet hum...",['Cats and Kindness Embrace'],"[""they didn't teach school to learn something ..."
3,3,18,3_water_open_cold_open water,"['water', 'open', 'cold', 'open water', 'condi...","['season swimming', 'swimming', 'swimmers', 's...",['Open Water Training Challenges'],['evening dawn in malo bali sets beautiful yet...


In [9]:
# Compare the topic embedding and the localized (keyword) embedding for one topic.
TOPIC_ID = 0 # case-insensitive substring

# Load the standard topic embeddings
topic_embeddings = json.loads((OUTPUT / "topic_embeddings.json").read_text())

# Load the localized topic embeddings (built from KeyBERT keywords)
keyword_embeddings = json.loads((OUTPUT / "topic_keyword_embeddings.json").read_text())

target_topic_id = TOPIC_ID

standard_topic_embedding = np.array(topic_embeddings[str(target_topic_id)], dtype=np.float32)
localized_topic_embedding = np.array(keyword_embeddings[str(target_topic_id)], dtype=np.float32)

cos = float(np.dot(standard_topic_embedding, localized_topic_embedding) /
            (np.linalg.norm(standard_topic_embedding) * np.linalg.norm(localized_topic_embedding)))
print(f"Topic {target_topic_id}: {labels[str(target_topic_id)]['label']}")
print(f"  cosine(centroid, localized) = {cos:.4f}")
print(f"  distance                    = {1 - cos:.4f}")
print("  → A larger distance means the keyword embedding is pulling the search")
print("    away from the centroid — useful when the centroid is diffuse.")

Topic 0: Space & Life Journeys
  cosine(centroid, localized) = 0.3014
  distance                    = 0.6986
  → A larger distance means the keyword embedding is pulling the search
    away from the centroid — useful when the centroid is diffuse.


# Coding Challenge: 
  Building on the code above, find the topic with the greatest difference (distance) between the topic
  embedding and the keyword embedding. 
  
  Which topic is it? 

  See ` solutions/challenge.py` for a solution

# Exercise 

The demo app has a **"use 'localized' search embedding"** toggle. Pick the topic
you found above (or any topic with high centroid-to-localized distance) and flip it.
Watch the search score and the result set shift toward more on-topic posts.

Review search results in the Demo App  
  -  Select the topic with the greatest difference (distance) between the topic
     embedding and the "localized" keyword embedding. 
  -  Toggle the `use "localized" search embedding` toggle. 
  - How do the results for the topics change? What happens to the search score? 
